In [ ]:
# Fig. 5c (as of 2025-05-27)

## Initialization

In [ ]:
# Imports
from datetime import datetime
from typing import Literal, Callable
from pathlib import Path
import math

import matplotlib.pyplot as plt
import matplotlib.ticker as plticker
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter, find_peaks

from data_processing import loading, types, helpers
from data_processing import processing as proc
from data_processing import dataframe_validation as df_valid
from data_processing import experiment_data_keys as edk
from data_processing.processing import neutron_window_strategy as nws

### Functions

In [ ]:
# fns ask questions, then generate strategy using factory

CalibrationKey = Literal[
    edk.ExperimentDataKey.CAEN_CALIBRATION,
    edk.ExperimentDataKey.NEW_CALIBRATION
]
NasaBorderKey = Literal[
    edk.ExperimentDataKey.NASA_BORDERS,
    edk.ExperimentDataKey.NASA_BORDERS_RECALC
]


def get_nasa_loading_settings(
    calib_key: CalibrationKey
) -> str:
    left_border_type = helpers.get_input_with_default(
        """\
Which left border calculation do you want to use?
1: original left border (0.1966 MeVee)
2: newer left border (~0.1866 MeVee)
3: CAEN lower limit (0.050 MeVee) (default)
Press Enter for default
""",
        3,
        int
    )
    border_key: NasaBorderKey = (
        edk.ExperimentDataKey.NASA_BORDERS if left_border_type == 1 
        else edk.ExperimentDataKey.NASA_BORDERS_RECALC
    )
    file_name_prefix = f"{calib_key.value}_{border_key.value}"
    return file_name_prefix


def get_n_distro_loading_settings(
    calib_key: CalibrationKey
) -> str:
    file_name_prefix = f"{calib_key.value}_{edk.ExperimentDataKey.N_WINDOW_BORDERS.value}"
    return file_name_prefix


def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> types.NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                edk.ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else edk.ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = loading.get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = loading.load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def get_n_distro_generation_settings(
) -> types.NeutronDistributionGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (3)
""",
        3,
        float
    )
    settings = types.NeutronDistributionGenerationSettings(
        sigma=sigma
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: nws.NeutronStrategyFactory,
    window_type: types.WindowType,
    loading: bool,
    settings: types.NeutronWindowSettings
) -> Callable[[], nws.AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: edk.ExperimentNeutronData, 
    factory_fn: Callable[[], nws.AbstractNeutronStrategy]
) -> edk.ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, edk.ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def correct_raw_signals(
    raw_signals_df: pd.DataFrame,
    baseline_idx_range: int = 40,
    baseline_offset: float = 0,
    max_adc: int = 16367,
    use_max_adc: bool = False
) -> pd.DataFrame:
    offset = int(baseline_offset * max_adc)
    signals_np = raw_signals_df.to_numpy()
    
    if use_max_adc:
        baselines = max_adc
    else:
        baselines = signals_np[
            :, :baseline_idx_range
        ].mean(axis=1).reshape(-1, 1)
    
    signals_np = -signals_np + baselines + offset
    corrected_signals = pd.DataFrame(
        signals_np,
        index=raw_signals_df.index,
        columns=raw_signals_df.columns
    )
    return corrected_signals

## Analysis

### User Inputs

In [ ]:
# experiment_ids = helpers.input_experiment_ids()
experiment_ids = ["TB-26"]

In [ ]:
experiment_neutron_data: edk.ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

### Something?

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    # exp_data[edk.ExperimentDataKey.UNCLASSIFIED] = loading.load_parquet_psd(exp_id)
    raw_psd_df = loading.load_caen_csvs(exp_id, get_flags=True, raw=True)
    print(exp_id, raw_psd_df.shape)
    print(raw_psd_df.head())
    exp_data["raw_psd_df"] = raw_psd_df
    # exp_data["raw_signals_df"] = loading.load_caen_csvs(
    #     exp_id, get_psd=False, get_signals=True, raw=True
    # )

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    signals_df = loading.load_caen_csvs(
        exp_id, get_psd=False, get_signals=True, raw=True
    )
    print(exp_id, signals_df.shape)
    exp_data["raw_signals_df"] = signals_df

In [ ]:
signals_count = 250
for exp_name, data_dict in experiment_neutron_data.items():
    raw_psd_df = data_dict["raw_psd_df"]
    raw_signals_df = data_dict["raw_signals_df"]

    clipped_idx = raw_psd_df["INPUT_SATURATING"].index
    clipped_signals = raw_signals_df.loc[clipped_idx].astype("int32")
    clipped_signals = clipped_signals.iloc[:signals_count]
    print(clipped_signals.shape)
    clipped_signals = correct_raw_signals(clipped_signals)
    print(clipped_signals.iloc[0, :])
    
    # raw_idx = raw_signals_df.index[:signals_count]
    # raw_signals = raw_signals_df.loc[raw_idx].astype("int32")
    # raw_signals.index = raw_signals.index.map(int)
    # raw_signals_peak = raw_signals.min(axis=1)
    # # raw_peak_heights = raw_signals_baseline - raw_signals_peak
    # raw_signals = raw_signals[raw_signals_peak > 0]
    # raw_signals = raw_signals.transpose()
    # raw_signals.index = raw_signals.index.map(int)
    # print(raw_signals.shape)
    # raw_signals_baseline = raw_signals.max()
    # raw_signals = -raw_signals + raw_signals_baseline

    data_dict["raw_signals"] = clipped_signals

## Plotting

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"

In [ ]:
alpha = 0.3

for exp_name, data_dict in experiment_neutron_data.items():
    print(exp_name)
    raw_signals = data_dict["raw_signals"]
    signal_x = None

    fig, ax = plt.subplots(figsize=(14, 7))
    for signal_id, signal_col in raw_signals.iterrows():
        if signal_x is None:
            signal_x = signal_col.index.map(int) * 2
        signal_y = signal_col.values
        ax.plot(
            signal_x,
            signal_y,
            alpha=alpha,
            color=bg_blue,
            lw=2
        )

    ax.tick_params(labelsize=fontsize)
    ax.set_xlabel("Time (ns)", fontsize=fontsize)
    ax.set_ylabel("Pulse height (ADC channels x1000)", fontsize=fontsize)
    # ax.xaxis.set_major_formatter(lambda x, _: f"{x * 2}")
    ax.yaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")

In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()